In [1]:
!pip install yfinance

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf

ticker = "LLY"

df = yf.download(
    ticker,
    start="2020-01-01",
    progress=False,
    actions=True,
    auto_adjust=False
)

df = df.reset_index()

# Flatten columns if yfinance returns a MultiIndex
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [col[0].lower().replace(" ", "_") for col in df.columns]
else:
    df.columns = [col.lower().replace(" ", "_") for col in df.columns]

df["ticker"] = ticker

# Standardize expected column names
rename_map = {
    "date": "date",
    "open": "open",
    "high": "high",
    "low": "low",
    "close": "close",
    "adj_close": "adj_close",
    "volume": "volume",
    "dividends": "dividends",
    "stock_splits": "stock_splits",
}

df = df.rename(columns=rename_map)

# Ensure corporate action columns exist
for col in ["dividends", "stock_splits"]:
    if col not in df.columns:
        df[col] = 0.0

df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

df["daily_return"] = df.groupby("ticker")["adj_close"].pct_change()
df["log_return"] = np.log(df["adj_close"] / df.groupby("ticker")["adj_close"].shift(1))
df["dollar_volume"] = df["close"] * df["volume"]

df["year"] = pd.to_datetime(df["date"]).dt.year
df["month"] = pd.to_datetime(df["date"]).dt.month
df["quarter"] = "Q" + pd.to_datetime(df["date"]).dt.quarter.astype(str)

panel = df[
    [
        "ticker",
        "date",
        "open",
        "high",
        "low",
        "close",
        "adj_close",
        "volume",
        "dividends",
        "stock_splits",
        "daily_return",
        "log_return",
        "dollar_volume",
        "year",
        "month",
        "quarter",
    ]
].copy()

panel.to_csv(
    "lly_daily_panel_2020_latest.csv",
    index=False,
    na_rep="N/A",
    date_format="%Y-%m-%d"
)

panel.head()

,ticker,date,open,high,low,close,adj_close,volume,dividends,stock_splits,daily_return,log_return,dollar_volume,year,month,quarter
0,LLY,2020-01-02,131.770004,132.259995,130.729996,132.210007,122.582100,2204200,0.0,0.0,NaN,NaN,2.914173e+08,2020,1,Q1
1,LLY,2020-01-03,130.300003,132.470001,130.229996,131.770004,122.174149,1963500,0.0,0.0,-0.003328,-0.003334,2.587304e+08,2020,1,Q1
2,LLY,2020-01-06,131.419998,132.559998,130.940002,132.259995,122.628433,2102900,0.0,0.0,0.003718,0.003711,2.781295e+08,2020,1,Q1
3,LLY,2020-01-07,131.699997,132.929993,131.699997,132.509995,122.860252,2448300,0.0,0.0,0.001890,0.001889,3.244242e+08,2020,1,Q1
4,LLY,2020-01-08,132.460007,134.210007,132.009995,133.710007,123.972878,5188600,0.0,0.0,0.009056,0.009015,6.937677e+08,2020,1,Q1


In [4]:
from google.colab import files

files.download("lly_daily_panel_2020_latest.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>